<a href="https://colab.research.google.com/github/seo104/ESAA/blob/main/Project3_%EB%B6%80%EC%8A%A4%ED%8C%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#부스팅 모델 학습 및 튜닝
##데이터 전처리

In [1]:
!wget -q https://raw.githubusercontent.com/sjlee53/ESAA-submission/YB/project/3/common.py

from common import *
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np

train, test, sub = load_data()

print("[Train 데이터]")
display(train.head())

print("[Test 데이터]")
display(test.head())

X, y, X_test = make_xy(train, test)

[Train 데이터]


,title,distributor,genre,release_time,time,screening_rat,director,dir_prev_bfnum,dir_prev_num,num_staff,num_actor,box_off_num
0,개들의 전쟁,롯데엔터테인먼트,액션,2012-11-22,96,청소년 관람불가,조병옥,NaN,0,91,2,23398
1,내부자들,(주)쇼박스,느와르,2015-11-19,130,청소년 관람불가,우민호,1161602.50,2,387,3,7072501
2,은밀하게 위대하게,(주)쇼박스,액션,2013-06-05,123,15세 관람가,장철수,220775.25,4,343,4,6959083
3,나는 공무원이다,(주)NEW,코미디,2012-07-12,101,전체 관람가,구자홍,23894.00,2,20,6,217866
4,불량남녀,쇼박스(주)미디어플렉스,코미디,2010-11-04,108,15세 관람가,신근호,1.00,1,251,2,483387


[Test 데이터]


,title,distributor,genre,release_time,time,screening_rat,director,dir_prev_bfnum,dir_prev_num,num_staff,num_actor
0,용서는 없다,시네마서비스,느와르,2010-01-07,125,청소년 관람불가,김형준,3.005290e+05,2,304,3
1,아빠가 여자를 좋아해,(주)쇼박스,멜로/로맨스,2010-01-14,113,12세 관람가,이광재,3.427002e+05,4,275,3
2,하모니,CJ 엔터테인먼트,드라마,2010-01-28,115,12세 관람가,강대규,4.206611e+06,3,419,7
3,의형제,(주)쇼박스,액션,2010-02-04,116,15세 관람가,장훈,6.913420e+05,2,408,2
4,평행 이론,CJ 엔터테인먼트,공포,2010-02-18,110,15세 관람가,권호영,3.173800e+04,1,380,1


##기본 모델 학습 및 성능 비교
XGBoost와 LightGBM의 기본 모델을 각각 학습

두 모델의 입력 데이터와 평가 방법을 동일하게 설정하여 성능 비교

###XGBoost 기본 모델

In [2]:
xgb_model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    verbosity=0
)

# 교차검증 성능 확인
xgb_score = evaluate(xgb_model, X, y)

# Test 데이터 예측
xgb_pred = fit_predict(xgb_model, X, y, X_test)

# 제출 파일 생성
save_submission(xgb_pred, sub, 'submission_xgboost.csv')

XGBRegressor             RMSE    1,534,035   (폴드 표준편차   240,468, 5-fold x 3seed)
저장: submission_xgboost.csv  (중앙값 13,777 / 평균 578,075 / 최대 15,327,377)


,title,box_off_num
0,용서는 없다,573324
1,아빠가 여자를 좋아해,667676
2,하모니,178676
3,의형제,1478625
4,평행 이론,1746266
...,...,...
238,해에게서 소년에게,232
239,울보 권투부,2591
240,어떤살인,39186
241,말하지 못한 비밀,2485


###LightGBM 기본 모델

In [3]:
lgbm_model = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

# 교차검증 성능 확인
lgbm_score = evaluate(lgbm_model, X, y)

# Test 데이터 예측
lgbm_pred = fit_predict(lgbm_model, X, y, X_test)

# 제출 파일 생성
save_submission(lgbm_pred, sub, 'submission_lightgbm.csv')

LGBMRegressor            RMSE    1,581,484   (폴드 표준편차   284,807, 5-fold x 3seed)
저장: submission_lightgbm.csv  (중앙값 14,507 / 평균 539,996 / 최대 13,816,997)


,title,box_off_num
0,용서는 없다,675949
1,아빠가 여자를 좋아해,602937
2,하모니,191467
3,의형제,868086
4,평행 이론,631583
...,...,...
238,해에게서 소년에게,256
239,울보 권투부,1508
240,어떤살인,42337
241,말하지 못한 비밀,1980


###성능 비교
| 비교 항목 | XGBoost | LightGBM |
|---|---|---|
| 교차검증 RMSE | 1,534,035 | 1,581,484 |
| LB 점수 | 1,123,657.10	 | 1,117,838.27 |
| 최대 예측 | 15,327,377 | 13,816,997 |

교차검증 결과는 XGBoost가 더 좋은 성능을 보였고, LB 점수는 LightGBM이 더 좋았다.

LB 점수를 기준으로 하여 LightGBM의 하이퍼파라미터 탐색

##하이퍼파라미터 튜닝
- n_estimators : 트리 개수 조정
- learning_rate : 학습 속도 조정
- num_leaves : 트리의 복잡도 조정
- max_depth : 트리 깊이 조정
- min_child_samples : 과적합 방지


###(1) 트리 복잡도 조정
기본 모델 : num_leaves=15, max_depth=-1

In [4]:
lgbm_1 = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=7,
    max_depth=3,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

# 교차검증
lgbm_score_1 = evaluate(lgbm_1, X, y)

# 예측 및 제출 파일 생성
lgbm_pred_1 = fit_predict(lgbm_1, X, y, X_test)
save_submission(lgbm_pred_1, sub, 'submission_lightgbm_1차.csv')

LGBMRegressor            RMSE    1,542,025   (폴드 표준편차   248,793, 5-fold x 3seed)
저장: submission_lightgbm_1차.csv  (중앙값 10,967 / 평균 479,766 / 최대 9,172,928)


,title,box_off_num
0,용서는 없다,471910
1,아빠가 여자를 좋아해,643630
2,하모니,258600
3,의형제,655487
4,평행 이론,968989
...,...,...
238,해에게서 소년에게,328
239,울보 권투부,3017
240,어떤살인,41514
241,말하지 못한 비밀,1679


교차검증 RMSE

    1,581,484 -> 1,542,025

LB 점수

    1,111,838.27 -> 1,330,097.46

LB 점수가 나빠졌으므로 트리 복잡도를 낮추는 것이 정확도를 높히지 않음.

###(2) 학습률과 트리 개수 조정
기본 모델 : learning_rate=0.05, n_estimators=600

In [5]:
lgbm_2 = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

# 교차검증
lgbm_score_2 = evaluate(lgbm_2, X, y)

# 예측 및 제출 파일 생성
lgbm_pred_2 = fit_predict(lgbm_2, X, y, X_test)
save_submission(lgbm_pred_2, sub, 'submission_lightgbm_2차.csv')

LGBMRegressor            RMSE    1,585,414   (폴드 표준편차   264,687, 5-fold x 3seed)
저장: submission_lightgbm_2차.csv  (중앙값 14,997 / 평균 534,931 / 최대 15,164,527)


,title,box_off_num
0,용서는 없다,553708
1,아빠가 여자를 좋아해,729730
2,하모니,225399
3,의형제,1180978
4,평행 이론,701859
...,...,...
238,해에게서 소년에게,230
239,울보 권투부,1650
240,어떤살인,40510
241,말하지 못한 비밀,2135


교차검증 RMSE

    1,581,484 -> 1,585,414

LB 점수

    1,111,838.27 -> 1,120,848.91

두 점수가 모두 상승함. 기본 모델 유지

###(3) 과적합 방지 및 정규화
기본 모델 : min_child_samples=20

In [6]:
lgbm_3 = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1
)

# 교차검증
lgbm_score_3 = evaluate(lgbm_3, X, y)

# 예측 및 제출 파일 생성
lgbm_pred_3 = fit_predict(lgbm_3, X, y, X_test)
save_submission(lgbm_pred_3, sub, 'submission_lightgbm_3차.csv')

LGBMRegressor            RMSE    1,516,913   (폴드 표준편차   242,781, 5-fold x 3seed)
저장: submission_lightgbm_3차.csv  (중앙값 14,384 / 평균 494,077 / 최대 8,162,185)


,title,box_off_num
0,용서는 없다,748131
1,아빠가 여자를 좋아해,733753
2,하모니,247080
3,의형제,1631236
4,평행 이론,1084205
...,...,...
238,해에게서 소년에게,207
239,울보 권투부,1985
240,어떤살인,22825
241,말하지 못한 비밀,2926


교차검증 RMSE

    1,581,484 -> 1,516,913

LB 점수

    1,111,838.27 -> 1,265,962.28

RMSE값은 낮아졌지만 LB 점수는 높아짐. -> 기각

###(4) 트리 복잡도 유지 + 약한 규제

In [7]:
lgbm_4 = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=20,
    subsample=1.0,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1
)

# 교차검증
lgbm_score_4 = evaluate(lgbm_4, X, y)

# 예측 및 제출 파일 생성
lgbm_pred_4 = fit_predict(lgbm_4, X, y, X_test)
save_submission(lgbm_pred_4, sub, 'submission_lightgbm_4차.csv')

LGBMRegressor            RMSE    1,535,997   (폴드 표준편차   251,638, 5-fold x 3seed)
저장: submission_lightgbm_4차.csv  (중앙값 15,528 / 평균 527,514 / 최대 13,041,931)


,title,box_off_num
0,용서는 없다,519674
1,아빠가 여자를 좋아해,707382
2,하모니,213880
3,의형제,1097184
4,평행 이론,732262
...,...,...
238,해에게서 소년에게,345
239,울보 권투부,1578
240,어떤살인,38285
241,말하지 못한 비밀,2405


교차검증 RMSE

    1,581,484 -> 1,535,997

LB 점수

    1,111,838.27 -> 	1,107,033.93

두개 모두 성능 향상됨.

###(5) 트리의 복잡도 상향
(4)모델에서 num_leaves를 15에서 20으로 조절

In [8]:
lgbm_5 = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=20,
    max_depth=-1,
    min_child_samples=20,
    subsample=1.0,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1
)

# 교차검증
lgbm_score_5 = evaluate(lgbm_5, X, y)

# 예측 및 제출 파일 생성
lgbm_pred_5 = fit_predict(lgbm_5, X, y, X_test)
save_submission(lgbm_pred_5, sub, 'submission_lightgbm_5차.csv')

LGBMRegressor            RMSE    1,567,251   (폴드 표준편차   271,503, 5-fold x 3seed)
저장: submission_lightgbm_5차.csv  (중앙값 17,840 / 평균 574,565 / 최대 16,158,720)


,title,box_off_num
0,용서는 없다,576025
1,아빠가 여자를 좋아해,750578
2,하모니,204567
3,의형제,1333444
4,평행 이론,556805
...,...,...
238,해에게서 소년에게,258
239,울보 권투부,1172
240,어떤살인,36989
241,말하지 못한 비밀,1671


교차검증 RMSE

    1,581,484 -> 1,567,251
LB 점수

    1,111,838.27 -> 	1,094,819.58

두 점수 모두 좋아짐. -> 최종 채택

##결론
XGBoost와 LightGBM의 LB를 비교한 결과 LightGBM의 점수가 더 좋아 이로 선정하였다.

LightGBM의 기본모델에서의 LB는 1,111,838.27였고, 하이퍼파라미터 조절(num_leaves=20, reg_alpha=0.5, reg_lambda=1.0)을 통해서 1,094,819.58까지 개선되었다.